In [1]:
# %%
%cd ..
import torch as tn
import torchtt as tntt
import teneva as ten
import numpy as np
from time import time
import json

from src.qtt_interpolation.utils import *
from src.qtt_interpolation.int_tools import *
import pandas as pd
import seaborn as sns
from metrics.mutils import*

/home/users/siddhartha.morales/TTinterpolation/QTT-polynomial-interpolation/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/nfs/users/siddhartha.morales/TTinterpolation/QTT-polynomial-interpolation


In [2]:
def rcf(cores):
    R = [1] + [c.shape[2] for c in cores[:-1]] + [1]
    return tntt._decomposition.rl_orthogonal( cores, R, False )[0]

def full_i(cores,indx):
    tfull = cores[0][0, :, :]
    for i in range(1, indx):
        tfull = tn.einsum('...i,ijk->...jk', tfull, cores[i])
    return tfull

def ensure_torch_rng(rng=None, device=None):
    if rng is None:
        if device is None:
            device = "cuda" if tn.cuda.is_available() else "cpu"
        rng = tn.Generator(device=device)
        rng.seed()  # nondeterministic, like np.random.default_rng()
    return rng

def marginal_probs(mps, Tprev, site):
    tensor = mps[site]
    ampls = tn.einsum('ni,iaj->naj', Tprev, tensor)
    probs = (ampls**2).sum(2)
    return probs

def sample_r(probs, rng, sentinel=-1):
    p = probs.clamp_min(0)
    sums = p.sum(dim=1); nz = sums > 0
    out = tn.full((p.size(0),), sentinel, device="cpu")
    if nz.any():
        if rng is None:
            rng = tn.Generator(device="cpu").seed()
        out[nz] = tn.multinomial(p[nz], 1, replacement=True, generator=rng).squeeze(1)
    return out

def tt_mc_sample(tt_cores, Nsamples, rng = None, return_joint=False):

    if rng == None:
        rng = ensure_torch_rng()

    mps = rcf(tt_cores)

    L = len(mps)
    d = mps[0].shape[1]

    #first site
    mps0 = full_i(mps,1)
    p0 = (mps0**2).sum(1)
    p0 = tn.clamp(p0, min=0)
    p0 /= p0.sum()
    first_samples = tn.multinomial(p0, num_samples=Nsamples,replacement=True, generator=rng)

    Tprev = mps0[first_samples]        # (Nsamples, D1)
    prob_prev = (Tprev**2).sum(1)        # current accumulated probability
    bitstrings = first_samples.reshape(1, Nsamples)

    # Middle sites
    for site in range(1, L - 1):
        probs_cur = marginal_probs(mps, Tprev, site)        # (Nsamples, d)
        probs_cond = probs_cur / prob_prev[:, None]
        samples_site = sample_r(probs_cond, rng)
        # Contract selected tensors
        Tsite = mps[site][:,samples_site,:]                     # (Nsamples, Dl, Dr)
        Tprev = tn.einsum('ni,inj->nj', Tprev, Tsite)       # (Nsamples, Dr)
        prob_prev = (Tprev**2).sum(1)
        bitstrings = tn.concatenate((bitstrings, samples_site.reshape(1, Nsamples)), axis=0)

    # Last site
    last_tensor = mps[-1]           # shape (d, Dl)
    ampls_last_all = tn.einsum('ni,iqj->nq', Tprev, last_tensor)  # (Nsamples, d)
    probs_last_all = ampls_last_all**2
    probs_cond_last = probs_last_all / prob_prev[:, None]
    last_samples = sample_r(probs_cond_last, rng)
    bitstrings = tn.concatenate((bitstrings, last_samples.reshape(1, Nsamples)), axis=0)

    if return_joint:
        # amplitude for chosen last outcome
        ampl_final = tn.einsum('ni,ni->n', Tprev, last_tensor[last_samples])
        joint_probs = ampl_final**2
        return bitstrings, joint_probs

    return bitstrings

def tt_mc_sample_bo(tt_cores, m , Nsamples, rng = None, return_joint=False):

    if rng == None:
        rng = ensure_torch_rng()

    mps = rcf(tt_cores)
    L = len(mps)
    d = mps[0].shape[1]

    #first m sites
    mps0 = full_i(mps,m).reshape(d**m,-1)
    p0 = (mps0**2).sum(1)
    p0 = tn.clamp(p0, min=0)
    p0 /= p0.sum()
    first_samples = tn.multinomial(p0, num_samples=Nsamples,replacement=True, generator=rng)
    
    Tprev = mps0[first_samples]        # (Nsamples, D1)
    prob_prev = (Tprev**2).sum(1)        # current accumulated probability
    bitstrings = first_samples.reshape(1, Nsamples)

    # Middle sites
    for site in range(m, L - 1):
        probs_cur = marginal_probs(mps, Tprev, site)        # (Nsamples, d)
        probs_cond = probs_cur / prob_prev[:, None]
        samples_site = sample_r(probs_cond, rng)
        # Contract selected tensors
        Tsite = mps[site][:,samples_site,:]                     # (Nsamples, Dl, Dr)
        Tprev = tn.einsum('ni,inj->nj', Tprev, Tsite)       # (Nsamples, Dr)
        prob_prev = (Tprev**2).sum(1)
        bitstrings = tn.concatenate((bitstrings, samples_site.reshape(1, Nsamples)), axis=0)
    
        # Last site
    last_tensor = mps[-1]           # shape (d, Dl)
    ampls_last_all = tn.einsum('ni,iqj->nq', Tprev, last_tensor)  # (Nsamples, d)
    probs_last_all = ampls_last_all**2
    probs_cond_last = probs_last_all / prob_prev[:, None]
    last_samples = sample_r(probs_cond_last, rng)
    bitstrings = tn.concatenate((bitstrings, last_samples.reshape(1, Nsamples)), axis=0)

    if return_joint:
        # amplitude for chosen last outcome
        ampl_final = tn.einsum('ni,ni->n', Tprev, last_tensor[last_samples])
        joint_probs = ampl_final**2
        return bitstrings, joint_probs

    return bitstrings

def bin_bits(A: tn.Tensor, m: int, msb_first: bool = True,
                                as_bool: bool = False) -> tn.Tensor:
    """
    A: (N, d) with A[:,0] in [0, 2**m - 1]
    Returns: (N, m + d - 1), first m columns are the binary bits of A[:,0].
    """
    N, d = A.shape
    x = A[:, 0].to(tn.long)                 # integers for bit ops

    shifts = (tn.arange(m-1, -1, -1, device=A.device) if msb_first
              else tn.arange(m, device=A.device))
    bits = ((x.unsqueeze(1) >> shifts) & 1)

    if as_bool:
        bits = bits.bool()
    else:
        bits = bits.to(A.dtype)                # keep original dtype (e.g., float)

    out = tn.empty((N, m + d - 1), dtype=bits.dtype, device=A.device)
    out[:, :m] = bits
    out[:, m:] = A[:, 1:].to(out.dtype)
    return out

In [35]:
alpha = 100
R = 0.125
ftn = lambda X,Y : circles(X,Y,alpha=alpha,r=R)
ften = lambda X: circle_ten(X,alpha=alpha,r=R)
# %% Coarse grid 

p_coarse = 8
num_points = 2**p_coarse
pmax=17

pfinal = 10
p=pfinal

# SVD coarse
t = time()
x = tn.linspace(0, 1, num_points + 1, dtype=tn.float64)[:-1]
y = tn.linspace(0, 1, num_points + 1, dtype=tn.float64)[:-1]
X, Y = tn.meshgrid(x, y, indexing='ij')

mask_grid = ftn(X, Y)
mask_svd = tntt.TT(zM(mask_grid,p_coarse),[2,2]*p_coarse,eps=1e-10)
t_mask = time() - t


ncoarse = [ 2,2]*p_coarse
m         = None  # Number of calls to target function
e         =  1e-7   # Desired accuracy
nswp      = 3   # Sweep number
r         = 30      # TT-rank of the initial tensor
dr_min    = 2      # Cross parameter (minimum number of added rows)
dr_max    = 5      # Cross parameter (maximum number of added rows)

f = lambda I: ften(ind_to_r_ten(I,a=0,b=1,d=2,pdp=1))

Yc = ten.rand(ncoarse, r)
cache,info_coarse = {}, {}
mask_ttc = ten.cross(f, Yc, m, e, nswp, dr_min=dr_min, dr_max=dr_max,log=True,cache=cache,m_cache_scale=5,info=info_coarse)
mask_ttc = ten.truncate(mask_ttc,1e-12)

# do the interpolation
t = time()
mask_svd_i = qtt_skcubic2d_p(mask_svd, pfinal, eps=1e-12, order=1).round(1e-3)
t_svd_i = time() - t
#mask_svd_i = [c.numpy() for c in mask_svd_i.cores]

t = time()
mask_ttc_i = qtt_skcubic2d_p( tntt.TT([tn.tensor(c,dtype=tn.float64) for c in mask_ttc]) ,pfinal, eps=1e-12, order=1).round(1e-3)
t_ttc_i = time() - t
#mask_ttc_i = [c.numpy() for c in mask_ttc_i.cores]

#cross over the fine grid

n = [ 2,2]*pfinal   # Shape of the tensor
m         = None  # Number of calls to target function
e         =  1e-7   # Desired accuracy
nswp      = 3   # Sweep number
r         = 30 + (p-p_coarse)*2      # TT-rank of the initial tensor
dr_min    = 2      # Cross parameter (minimum number of added rows)
dr_max    = 5      # Cross parameter (maximum number of added rows)

Y = ten.rand(n, r)
info_fine,cache = {}, {}
Y = ten.cross(f, Y, m, e, nswp, dr_min=dr_min, dr_max=dr_max, cache=cache,m_cache_scale=5,log=True,info=info_fine)
Yr = ten.truncate(Y, 1e-3) 

''' 
#validate over 1e5 points
m_tst = int(1e5)
# Random multi-indices for the test points:
I_tst = np.vstack([np.random.choice(k, m_tst) for k in n]).T
y_tst = f(I_tst)
norm = np.linalg.norm(y_tst)

y_tt = ten.get_many(Yr, I_tst)
e_tst = np.linalg.norm(y_tt - y_tst) / norm

y_tti = ten.get_many(mask_ttc_i, I_tst)
e_tstti = np.linalg.norm(y_tti - y_tst) / norm

y_tti_svd = ten.get_many(mask_svd_i, I_tst)
e_tstti_svd = np.linalg.norm(y_tti - y_tst) / norm
print('errors', e_tst,e_tstti,e_tstti_svd)
'''
print('doing errors...',flush=True)
N = 2**pfinal
x = tn.linspace(0, 1, N + 1, dtype=tn.float64)[:-1]
y = tn.linspace(0, 1, N + 1, dtype=tn.float64)[:-1]
X, Y = tn.meshgrid(x, y, indexing='ij')
mask_grid = ftn(X, Y)
mask_full = zM(mask_grid,pfinal)
norm = tn.linalg.norm(mask_grid)


y_tt = tntt.TT( [tn.tensor(c,dtype=tn.float64) for c in Yr] ).full().reshape(N,N)
e_tst = tn.linalg.norm(mask_full - y_tt) / norm

y_tti = mask_ttc_i.full().reshape(N,N)
e_tstti = tn.linalg.norm(mask_full - y_tti) / norm

y_tti_svd = mask_svd_i.full().reshape(N,N)
e_tstti_svd = tn.linalg.norm(mask_full - y_tti_svd) / norm

# pre | time:      0.831 | evals: 0.00e+00 (+ 0.00e+00) | rank:  30.0 | 
#   1 | time:      2.892 | evals: 1.21e+04 (+ 2.00e+04) | rank:  26.6 | e: 1.0e+00 | 
#   2 | time:      6.104 | evals: 1.75e+04 (+ 5.99e+04) | rank:  31.0 | e: 3.2e-03 | 
#   3 | time:     10.940 | evals: 2.25e+04 (+ 1.16e+05) | rank:  35.7 | e: 2.7e-06 | stop: conv | 
# pre | time:      1.436 | evals: 0.00e+00 (+ 0.00e+00) | rank:  34.0 | 
#   1 | time:      5.542 | evals: 2.55e+04 (+ 3.69e+04) | rank:  32.6 | e: 1.0e+00 | 
#   2 | time:     12.382 | evals: 4.19e+04 (+ 1.11e+05) | rank:  38.7 | e: 4.6e-03 | 
#   3 | time:     22.526 | evals: 5.89e+04 (+ 2.17e+05) | rank:  44.5 | e: 9.6e-04 | stop: nswp | 
doing errors...


In [4]:
print(e_tst,e_tstti_svd,e_tstti)

tensor(0.0008, dtype=torch.float64) tensor(0.0015, dtype=torch.float64) tensor(0.0015, dtype=torch.float64)


In [58]:
Nsamples = int(1e6)
samp = tt_mc_sample(mask_svd_i.cores, Nsamples, rng = None, return_joint=False)
vals = mask_svd_i.apply_mask(samp.T)

In [61]:
Nsamples = int(1e6)
ibs = 16
samp = tt_mc_sample_bo(mask_svd_i.cores,ibs, Nsamples, rng = None, return_joint=False)
samp = bin_bits(samp.T,ibs).T
vals = mask_svd_i.apply_mask(samp.T)

In [62]:
x = (samp.T[:,0::2] * tn.tensor([2**(-i-1) for i in range(pfinal)])).sum(1)
y = (samp.T[:,1::2] * tn.tensor([2**(-1-i) for i in range(pfinal)])).sum(1)

pts = tn.stack([x,y], axis=-1)
pts

tensor([[0.4121, 0.4980],
        [0.4727, 0.5225],
        [0.4346, 0.5615],
        ...,
        [0.4316, 0.5410],
        [0.4980, 0.5264],
        [0.4014, 0.4219]])

In [63]:
((circles(x,y)-vals)**2/Nsamples).sum().sqrt()

tensor(0.0014, dtype=torch.float64)